In [2]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

In [3]:
import pandas as pd

df_layoffs = pd.read_csv('../data/layoffs_events.csv')
df_news = pd.read_csv('../data/news_sentiment.csv')

print("Layoffs shape:", df_layoffs.shape)
print("\nLayoffs columns:", df_layoffs.columns.tolist())
print("\nLayoffs sample:")
print(df_layoffs.head(3))

print("\n---")
print("News shape:", df_news.shape)
print("\nNews columns:", df_news.columns.tolist())
print("\nNews sample:")
print(df_news.head(3))

Layoffs shape: (2470, 11)

Layoffs columns: ['company', 'location', 'layoff_count', 'date', 'pct_workforce', 'industry', 'source_url', 'stage', 'raised_mm', 'country', 'is_ai_company']

Layoffs sample:
        company     location  layoff_count        date pct_workforce  \
0   Panda Squad  SF Bay Area           6.0  2020-03-13           75%   
1  HopSkipDrive  Los Angeles           8.0  2020-03-13           10%   
2      Help.com       Austin          16.0  2020-03-16          100%   

       industry                                         source_url    stage  \
0      Consumer  https://twitter.com/danielsing er/status/12385...     Seed   
1  Transportat…  https://layoffs.fyi/2020/04/02/ hopskipdrive-l...  Unknown   
2       Support                                           LinkedIn     Seed   

  raised_mm        country  is_ai_company  
0    $1.00   United States          False  
1   $45.00   United States          False  
2    $6.00   United States          False  

---
News shape:

In [4]:
def create_layoff_chunk(row):
    company = row['company'] if pd.notna(row['company']) else 'Unknown company'
    location = row['location'] if pd.notna(row['location']) else 'Unknown location'
    count = f"{int(row['layoff_count'])} employees" if pd.notna(row['layoff_count']) else 'an unknown number of employees'
    date = row['date'] if pd.notna(row['date']) else 'unknown date'
    pct = f"({row['pct_workforce']} of workforce)" if pd.notna(row['pct_workforce']) else ''
    industry = row['industry'] if pd.notna(row['industry']) else 'Unknown'
    country = row['country'] if pd.notna(row['country']) else 'Unknown'
    ai = "an AI company" if row['is_ai_company'] == True else "a non-AI company"
    
    return f"{company}, based in {location} ({country}), laid off {count} {pct} on {date}. They operate in the {industry} industry and are {ai}."

df_layoffs['date'] = pd.to_datetime(df_layoffs['date']).dt.strftime('%B %d, %Y')
df_layoffs['chunk'] = df_layoffs.apply(create_layoff_chunk, axis=1)

print("Sample chunks:")
for chunk in df_layoffs['chunk'].head(3):
    print(chunk)
    print()

Sample chunks:
Panda Squad, based in SF Bay Area (United States), laid off 6 employees (75% of workforce) on March 13, 2020. They operate in the Consumer industry and are a non-AI company.

HopSkipDrive, based in Los Angeles (United States), laid off 8 employees (10% of workforce) on March 13, 2020. They operate in the Transportat… industry and are a non-AI company.

Help.com, based in Austin (United States), laid off 16 employees (100% of workforce) on March 16, 2020. They operate in the Support industry and are a non-AI company.



In [5]:
def create_news_chunk(row):
    date = row['date'] if pd.notna(row['date']) else 'unknown date'
    title = row['title'] if pd.notna(row['title']) else 'Unknown title'
    source = row['source'] if pd.notna(row['source']) else 'Unknown source'
    sentiment = row['sentiment_cat'] if pd.notna(row['sentiment_cat']) else 'unknown'
    description = row['description'] if pd.notna(row['description']) else ''
    
    chunk = f"On {date}, {source} published: '{title}'. The sentiment of this article was {sentiment}."
    if description:
        chunk += f" Description: {description}"
    
    return chunk

df_news['chunk'] = df_news.apply(create_news_chunk, axis=1)

print("Sample news chunks:")
for chunk in df_news['chunk'].head(3):
    print(chunk)
    print()

Sample news chunks:
On 2022-11-09, Reuters published: 'Twitter lays off roughly half its workforce after Elon Musk takeover'. The sentiment of this article was neutral.

On 2022-11-09, NYT published: 'Meta to cut more than 11,000 employees in biggest layoff in company history'. The sentiment of this article was negative.

On 2022-11-14, WSJ published: 'Amazon begins largest round of layoffs in company history'. The sentiment of this article was positive.



In [6]:
# Combine both datasets into one list of documents
layoff_chunks = df_layoffs['chunk'].tolist()
news_chunks = df_news['chunk'].tolist()

all_chunks = layoff_chunks + news_chunks

# Create metadata for each chunk so we know where it came from
layoff_metadata = [{"source": "layoffs", "index": i} for i in range(len(layoff_chunks))]
news_metadata = [{"source": "news", "index": i} for i in range(len(news_chunks))]

all_metadata = layoff_metadata + news_metadata

# Create unique IDs for each chunk
all_ids = [f"layoff_{i}" for i in range(len(layoff_chunks))] + \
          [f"news_{i}" for i in range(len(news_chunks))]

print(f"Total layoff chunks: {len(layoff_chunks)}")
print(f"Total news chunks: {len(news_chunks)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"\nSample ID: {all_ids[0]}")
print(f"Sample metadata: {all_metadata[0]}")

Total layoff chunks: 2470
Total news chunks: 306
Total chunks: 2776

Sample ID: layoff_0
Sample metadata: {'source': 'layoffs', 'index': 0}
